In [ ]:
!pip install "numpy<2.0"

In [ ]:
!pip install transformers==4.52.4 simpletransformers==0.70.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 135.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.3/316.3 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 131.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 95.6 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=0f322a1762088b191fe3117418f715fdb0951a35d39791dbac61164b10fb97f6
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
  Attempting uninstall: tokenizers
    

In [1]:
%pip install shap

In [2]:
import shap

shap.__version__

'0.50.0'

In [ ]:
%pip show simpletransformers

Name: simpletransformers
Version: 0.70.1
Summary: An easy-to-use wrapper library for the Transformers library.
Home-page: https://github.com/ThilinaRajapakse/simpletransformers/
Author: Thilina Rajapakse
Author-email: chaturangarajapakshe@gmail.com
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: datasets, numpy, pandas, regex, requests, scikit-learn, scipy, sentencepiece, seqeval, streamlit, tensorboard, tensorboardx, tokenizers, tqdm, transformers, wandb
Required-by: 


In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np

# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel('/content/drive/My Drive/SKR/final_train_test (2).xlsx')

In [ ]:
df['annotation'] = df['annotation'].replace('Neutral', 'N')
model_df = df[['text', 'annotation']].copy()
model_df['labels'] = model_df['annotation'].map({'Y': 1, 'N': 0})
model_df = model_df[['text', 'labels']]

In [ ]:
from sklearn.model_selection import train_test_split

train_df, eval_df = train_test_split(
    model_df,
    test_size=0.215,       # 20% of the data will be used for testing
    random_state=42,     # Ensures the split is the same every time
    stratify=df['annotation'] # Ensures train and test sets have the same proportion of labels
)

# You can check the size of your new dataframes
print(f"Training set size: {len(train_df)}")
print(f"Testing set size: {len(eval_df)}")

Training set size: 3045
Testing set size: 835


###  DistillBERT

In [ ]:
from simpletransformers.classification import ClassificationModel, ClassificationArgs
import logging

# Optional: Configure logging
logging.basicConfig(level=logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)
# Define model arguments
model_args = ClassificationArgs()
model_args.num_train_epochs = 5
model_args.overwrite_output_dir = True
model_args.manual_seed = 42
model_args.use_multiprocessing = False # Set to False for notebook environments
model_args.evaluate_during_training = True

model = ClassificationModel(
    "distilbert",
    "distilbert/distilbert-base-cased",
    num_labels=2,
    args=model_args,
    use_cuda=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
model.train_model(train_df, eval_df=eval_df)

Epoch:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:882: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler()


Running Epoch 1 of 5:   0%|          | 0/381 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:905: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


  0%|          | 0/1 [00:00<?, ?it/s]

...

Running Epoch 5 of 5:   0%|          | 0/381 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:905: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


  0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:1505: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


(1905,
 defaultdict(list,
             {'global_step': [381, 762, 1143, 1524, 1905],
              'train_loss': [0.01865215227007866,
               0.0032655715476721525,
               0.012286663055419922,
               0.010328483767807484,
               0.00040950774564407766],
              'mcc': [np.float64(0.8331011568483644),
               np.float64(0.8561851650447081),
               np.float64(0.8779254410412068),
               np.float64(0.890860337013183),
               np.float64(0.882721811991294)],
              'accuracy': [0.9161676646706587,
               0.9281437125748503,
               0.9389221556886228,
               0.9449101796407186,
               0.9413173652694611],
              'f1_score': [0.9161242365798778,
               0.9280445348316155,
               0.9388533989239577,
               0.9448873354224778,
               0.9412513048485083],
              'tp': [np.int64(373),
               np.int64(372),
               np.int64(378),


In [ ]:
result, model_outputs, wrong_predictions = model.eval_model(eval_df)

print("\nEvaluation Results:")
print(result)

  0%|          | 0/1 [00:00<?, ?it/s]

Running Evaluation:   0%|          | 0/9 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:1505: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():



Evaluation Results:
{'mcc': np.float64(0.882721811991294), 'accuracy': 0.9413173652694611, 'f1_score': 0.9412513048485083, 'tp': np.int64(379), 'tn': np.int64(407), 'fp': np.int64(29), 'fn': np.int64(20), 'auroc': np.float64(0.9862960152675267), 'auprc': np.float64(0.9848474978250213), 'eval_loss': 0.32352181606822544}


### Roberta

In [ ]:
from simpletransformers.classification import ClassificationModel, ClassificationArgs
import logging

# Optional: Configure logging
logging.basicConfig(level=logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)
# Define model arguments
model_args = ClassificationArgs()
model_args.num_train_epochs = 5
model_args.overwrite_output_dir = True
model_args.manual_seed = 42
model_args.use_multiprocessing = False # Set to False for notebook environments
model_args.evaluate_during_training = True

model = ClassificationModel(
    "roberta",
    "roberta-base",
    num_labels=2,
    args=model_args,
    use_cuda=True
)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
model.train_model(train_df, eval_df=eval_df)

Epoch:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:882: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler()


Running Epoch 1 of 5:   0%|          | 0/322 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:905: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


  0%|          | 0/1 [00:00<?, ?it/s]

...

Running Epoch 5 of 5:   0%|          | 0/322 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:905: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


  0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:1505: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


(1610,
 defaultdict(list,
             {'global_step': [322, 644, 966, 1288, 1610],
              'train_loss': [0.8819134831428528,
               0.010091781616210938,
               0.0033828418236225843,
               0.0005761782522313297,
               0.0004769960942212492],
              'mcc': [np.float64(0.7249249956435401),
               np.float64(0.8676003349087634),
               np.float64(0.8792165640803208),
               np.float64(0.8981952668245847),
               np.float64(0.8846513092746353)],
              'accuracy': [0.8482269503546099,
               0.9333333333333333,
               0.9390070921985816,
               0.948936170212766,
               0.9418439716312057],
              'f1_score': [0.8447165333097291,
               0.9332559839983563,
               0.9389240232175344,
               0.9489064613798879,
               0.9417764966794172],
              'tp': [np.int64(352),
               np.int64(341),
               np.int64(344),
 

In [ ]:
result, model_outputs, wrong_predictions = model.eval_model(eval_df)

print("\nEvaluation Results:")
print(result)

  0%|          | 0/1 [00:00<?, ?it/s]

Running Evaluation:   0%|          | 0/8 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:1505: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():



Evaluation Results:
{'mcc': np.float64(0.8846513092746353), 'accuracy': 0.9418439716312057, 'f1_score': 0.9417764966794172, 'tp': np.int64(344), 'tn': np.int64(320), 'fp': np.int64(29), 'fn': np.int64(12), 'auroc': np.float64(0.9887559962654132), 'auprc': np.float64(0.9887310191311045), 'eval_loss': 0.4682918172329664}


### ModernBERT

In [ ]:

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

model_name = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,  # Binary classification
    problem_type="single_label_classification"
)

# Custom Dataset class
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])
        label = self.labels.iloc[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Create datasets
train_dataset = TextClassificationDataset(
    train_df['text'],
    train_df['labels'],
    tokenizer
)

eval_dataset = TextClassificationDataset(
    eval_df['text'],
    eval_df['labels'],
    tokenizer
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=32,  # Increased from 16 if you have enough GPU memory
    per_device_eval_batch_size=64,
    warmup_steps=100,  # Reduced from 500
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,  # More frequent logging to see progress
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,  # Reduced to save space
    seed=42,
    fp16=torch.cuda.is_available(),  # Enable mixed precision only if CUDA available
    dataloader_num_workers=4,  # Parallel data loading
    gradient_accumulation_steps=1,
    # report_to=None,  # Disable reporting to prevent wandb issues
)


# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-2240314284.py:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
W0825 04:43:02.333000 1420 torch/_inductor/utils.py:1436] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.620200,0.199472,0.923353,0.922929,0.928235,0.923353
2,0.164300,0.110231,0.965269,0.965263,0.965282,0.965269
3,0.034100,0.140043,0.970060,0.970061,0.970066,0.970060
4,0.016200,0.147276,0.968862,0.968841,0.969099,0.968862
5,0.000300,0.140195,0.973653,0.973655,0.973669,0.973653


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: T

TrainOutput(global_step=480, training_loss=0.1281239094833533, metrics={'train_runtime': 765.5755, 'train_samples_per_second': 19.887, 'train_steps_per_second': 0.627, 'total_flos': 5188038205593600.0, 'train_loss': 0.1281239094833533, 'epoch': 5.0})

In [ ]:
print("Final evaluation:")
print(trainer.evaluate())

Final evaluation:


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


{'eval_loss': 0.14019542932510376, 'eval_accuracy': 0.9736526946107784, 'eval_f1': 0.9736553444834469, 'eval_precision': 0.973669182544618, 'eval_recall': 0.9736526946107784, 'eval_runtime': 9.8901, 'eval_samples_per_second': 84.428, 'eval_steps_per_second': 1.416, 'epoch': 5.0}


In [ ]:
model.save_pretrained("./fine_tuned_modernbert")
tokenizer.save_pretrained("./fine_tuned_modernbert")

# INFERENCE

In [ ]:
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import average_precision_score
from scipy.special import softmax

# --- 1. Load your saved model and tokenizer ---
model_path = "/content/drive/My Drive/SKR/"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

eval_args = TrainingArguments(
    output_dir="./temp_eval",
    per_device_eval_batch_size=64,
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=eval_args,
    tokenizer=tokenizer
)

print("Running predictions on the evaluation set...")
predictions_output = trainer.predict(eval_dataset)

# 'predictions' contains the raw logits (raw scores) from the model
# It's an array of shape (num_samples, 2)
logits = predictions_output.predictions
# 'label_ids' contains the true labels
labels = predictions_output.label_ids


try:
    probs = softmax(logits, axis=1)
    positive_class_scores = probs[:, 1]
except Exception as e:
    print(f"Error processing logits: {e}")
    # Handle case if logits are 1D (shouldn't be for num_labels=2)
    positive_class_scores = probs

# average_precision_score is scikit-learn's function for AUPRC
auprc = average_precision_score(labels, positive_class_scores)

print(f"\n--- Evaluation Results ---")
print(f"AUPRC (Average Precision): {auprc:.4f}")

HFValidationError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: './fine_tuned_modernbert'.

In [ ]:
predictions = trainer.predict(eval_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = eval_df['labels'].values

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [ ]:
# Function to predict on new text
def predict_text(text, model, tokenizer, device='cuda' if torch.cuda.is_available() else 'cpu'):
    model.eval()
    model.to(device)

    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        padding='max_length',
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_class = torch.argmax(probabilities, dim=-1).item()

    return {
        'predicted_class': predicted_class,
        'predicted_label': 'Y' if predicted_class == 1 else 'N',
        'confidence': probabilities[0][predicted_class].item(),
        'probabilities': {
            'N': probabilities[0][0].item(),
            'Y': probabilities[0][1].item()
        }
    }


In [ ]:
sample_text = "The breakfast is awesome, staff really friendly. I will come back here again if im here again!"
result = predict_text(sample_text, model, tokenizer)
result

{'predicted_class': 1,
 'predicted_label': 'Y',
 'confidence': 0.9999939203262329,
 'probabilities': {'N': 6.120221314631635e-06, 'Y': 0.9999939203262329}}